In [2]:
# ---------- SETTINGS ----------
STUDENT_FILE = "students.txt"
COURSE_FILE = "courses.txt"
REG_FILE = "registrations.txt"
MAX_CREDITS = 18


# ---------- FILE FUNCTIONS ----------
def read_lines(filename):
    """Read a file. Return a list of lines, and each line is split into a list.
    Example: "CS101,Programming,3,30" becomes ["CS101", "Programming", "3", "30"]"""
    lines = []
    file = None
    try:
        file = open(filename, "r")
        for line in file:
            line = line.strip()
            if line != "":
                lines.append(line.split(","))
    except FileNotFoundError:
        print(filename, "not found. Starting with empty data.")
    except PermissionError:
        print("No permission to read", filename)
    finally:
        if file != None:
            file.close()
    return lines


def save_all(students, courses, registrations):
    """Write all the data from memory into the three files."""
    try:
        file = open(STUDENT_FILE, "w")
        for sid in students:
            file.write(sid + "," + students[sid]["name"] + "," + students[sid]["password"] + "\n")
        file.close()

        file = open(COURSE_FILE, "w")
        for code in courses:
            info = courses[code]          # info is a tuple: (title, credits, seats)
            file.write(code + "," + info[0] + "," + str(info[1]) + "," + str(info[2]) + "\n")
        file.close()

        file = open(REG_FILE, "w")
        for sid in registrations:         # nested for loop
            for code in registrations[sid]:
                file.write(sid + "," + code + "\n")
        file.close()
    except PermissionError:
        print("Cannot save: no permission to write the files.")


def load_data():
    """Read the files and build three dictionaries. Returns all three (a tuple)."""
    students = {}        # {"S001": {"name": "Ayesha Khan", "password": "pass123"}}
    courses = {}         # {"CS101": ("Programming Fundamentals", 3, 30)}  -> (title, credits, seats)
    registrations = {}   # {"S001": {"CS101", "MG201"}}  -> a SET of course codes per student

    try:
        for row in read_lines(STUDENT_FILE):
            students[row[0]] = {"name": row[1], "password": row[2]}

        for row in read_lines(COURSE_FILE):
            courses[row[0]] = (row[1], int(row[2]), int(row[3]))

        for row in read_lines(REG_FILE):
            sid = row[0]
            code = row[1]
            if sid not in registrations:
                registrations[sid] = set()
            registrations[sid].add(code)
    except ValueError:
        print("A file has a number in the wrong format.")
    except IndexError:
        print("A file has a broken line.")

    return students, courses, registrations


def make_sample_data(students, courses):
    """If the files were empty, add a few sample students and courses."""
    if len(courses) == 0:
        courses["CS101"] = ("Programming Fundamentals", 3, 3)
        courses["MG201"] = ("Principles of Management", 3, 30)
        courses["ST301"] = ("Statistics for Analytics", 3, 30)
        courses["MK210"] = ("Marketing Basics", 2, 25)
    if len(students) == 0:
        students["S001"] = {"name": "Ayesha Khan", "password": "pass123"}
        students["S002"] = {"name": "Bilal Ahmed", "password": "pass456"}


# ---------- HELPER FUNCTIONS ----------
def count_seats_used(code, registrations):
    """Count how many students registered in one course."""
    count = 0
    for sid in registrations:
        if code in registrations[sid]:
            count = count + 1
    return count


def total_credits(sid, courses, registrations):
    """Add the credit hours of all courses of one student."""
    total = 0
    if sid in registrations:
        for code in registrations[sid]:
            total = total + courses[code][1]     # [1] is the credits inside the tuple
    return total


# ---------- 1) LOGIN ----------
def user_login(role, username, password, students):
    """Return True if the login is correct, otherwise False."""
    if role == "admin":
        return username == "admin" and password == "admin123"
    if username in students and students[username]["password"] == password:
        return True
    return False


# ---------- 2) ADD RECORDS ----------
def add_record(records, key, value):
    """Add key and value to a dictionary only if the key is new. Returns True or False."""
    if key in records:
        return False
    records[key] = value
    return True


def add_student(students, sid, name, password):
    """Add a student. Returns a message."""
    if sid == "" or name == "" or password == "":
        return "Error: nothing can be empty."
    if "," in sid + name + password:
        return "Error: do not use commas."
    if add_record(students, sid, {"name": name, "password": password}):
        return "Student " + name + " added."
    return "Error: ID " + sid + " already exists."


def add_course(courses, code, title, credits, seats):
    """Add a course. credits and seats come as text, so we convert them with int()."""
    try:
        credits = int(credits)
        seats = int(seats)
    except ValueError:
        return "Error: credits and seats must be numbers."
    if credits <= 0 or seats <= 0:
        return "Error: credits and seats must be more than 0."
    if code == "" or title == "" or "," in code + title:
        return "Error: code/title cannot be empty or have commas."
    if add_record(courses, code, (title, credits, seats)):
        return "Course " + code + " added."
    return "Error: course " + code + " already exists."


# ---------- 3) SEARCH ----------
def search_record(kind, students, courses, text):
    """Search students by name or courses by title. Returns a list of results.
    kind is "student" or "course"."""
    results = []
    text = text.lower()
    if kind == "student":
        for sid in students:
            if text in students[sid]["name"].lower():
                results.append(sid + " - " + students[sid]["name"])
    else:
        for code in courses:
            if text in courses[code][0].lower():
                results.append(code + " - " + courses[code][0])
    return results


# ---------- 4) REGISTER AND DROP ----------
def register_course(sid, code, students, courses, registrations):
    """Register a student in a course after checking the rules. Returns a message."""
    if sid not in students:
        return "Error: student not found."
    if code not in courses:
        return "Error: course " + code + " does not exist."

    if sid in registrations and code in registrations[sid]:
        return "Error: already registered in " + code + "."

    if count_seats_used(code, registrations) >= courses[code][2]:
        return "Error: " + code + " is full."

    new_total = total_credits(sid, courses, registrations) + courses[code][1]
    if new_total > MAX_CREDITS:
        return "Error: that would be " + str(new_total) + " credits. Limit is " + str(MAX_CREDITS) + "."

    if sid not in registrations:
        registrations[sid] = set()
    registrations[sid].add(code)
    save_all(students, courses, registrations)
    return "Success: registered in " + code + " - " + courses[code][0] + "."


def drop_course(sid, code, students, courses, registrations):
    """Remove a course from a student. Returns a message."""
    if sid in registrations and code in registrations[sid]:
        registrations[sid].remove(code)
        save_all(students, courses, registrations)
        return "Course " + code + " dropped."
    return "Error: you are not registered in " + code + "."


# ---------- 5) REPORT ----------
def generate_report(students, courses, registrations):
    """Make a text report: every course, its seats, and the students inside it."""
    report = "=== COURSE REGISTRATION REPORT ===\n"
    for code in courses:                                   # outer loop: courses
        used = count_seats_used(code, registrations)
        report = report + code + " - " + courses[code][0] + ": " + str(used) + "/" + str(courses[code][2]) + " seats filled\n"
        for sid in registrations:                          # inner loop: students
            if code in registrations[sid]:
                report = report + "     - " + sid + " " + students[sid]["name"] + "\n"
    return report


# ---------- SHOW COURSES ----------
def show_courses(courses, registrations):
    """Print all courses."""
    print("\nCODE    TITLE                         CREDITS  SEATS LEFT")
    for code in courses:
        info = courses[code]
        left = info[2] - count_seats_used(code, registrations)
        print(code, "|", info[0], "|", info[1], "credits |", left, "seats left")


# ---------- MENUS ----------
def admin_menu(students, courses, registrations):
    while True:
        print("\n--- ADMIN MENU ---")
        print("1. Add student")
        print("2. Add course")
        print("3. View courses")
        print("4. Search")
        print("5. Generate report")
        print("6. Logout")
        choice = input("Choose: ")

        if choice == "1":
            sid = input("Student ID: ")
            name = input("Name: ")
            password = input("Password: ")
            print(add_student(students, sid, name, password))
            save_all(students, courses, registrations)
        elif choice == "2":
            code = input("Course code: ")
            title = input("Course title: ")
            credits = input("Credit hours: ")
            seats = input("Seats: ")
            print(add_course(courses, code, title, credits, seats))
            save_all(students, courses, registrations)
        elif choice == "3":
            show_courses(courses, registrations)
        elif choice == "4":
            kind = input("Type 'student' or 'course': ")
            text = input("Search text: ")
            results = search_record(kind, students, courses, text)
            for item in results:
                print(item)
            if len(results) == 0:
                print("Nothing found.")
        elif choice == "5":
            report = generate_report(students, courses, registrations)
            print(report)
            try:
                file = open("report.txt", "w")
                file.write(report)
                file.close()
                print("(Report saved in report.txt)")
            except PermissionError:
                print("Could not save report.txt")
        elif choice == "6":
            print("Logged out.")
            break
        else:
            print("Please type a number from 1 to 6.")


def student_menu(sid, students, courses, registrations):
    print("\nWelcome,", students[sid]["name"])
    while True:
        print("\n--- STUDENT MENU ---")
        print("1. View courses")
        print("2. Search courses")
        print("3. Register in a course")
        print("4. Drop a course")
        print("5. My courses")
        print("6. Logout")
        choice = input("Choose: ")

        if choice == "1":
            show_courses(courses, registrations)
        elif choice == "2":
            text = input("Search text: ")
            results = search_record("course", students, courses, text)
            for item in results:
                print(item)
            if len(results) == 0:
                print("Nothing found.")
        elif choice == "3":
            code = input("Course code: ")
            print(register_course(sid, code, students, courses, registrations))
        elif choice == "4":
            code = input("Course code to drop: ")
            print(drop_course(sid, code, students, courses, registrations))
        elif choice == "5":
            if sid in registrations:
                for code in registrations[sid]:
                    print(code, "-", courses[code][0])
            print("Total credits:", total_credits(sid, courses, registrations))
        elif choice == "6":
            print("Logged out.")
            break
        else:
            print("Please type a number from 1 to 6.")


# ---------- MAIN PROGRAM ----------
def main():
    students, courses, registrations = load_data()
    make_sample_data(students, courses)
    save_all(students, courses, registrations)
    print("Loaded", len(students), "students and", len(courses), "courses.")

    while True:
        print("\n===== COURSE REGISTRATION PORTAL =====")
        print("1. Admin login")
        print("2. Student login")
        print("3. Exit")
        choice = input("Choose: ")

        if choice == "3":
            print("Goodbye!")
            break
        elif choice == "1" or choice == "2":
            username = input("Username / Student ID: ")
            password = input("Password: ")
            if choice == "1":
                if user_login("admin", username, password, students):
                    admin_menu(students, courses, registrations)
                else:
                    print("Wrong login.")
            else:
                if user_login("student", username, password, students):
                    student_menu(username, students, courses, registrations)
                else:
                    print("Wrong ID or password.")
        else:
            print("Please type 1, 2 or 3.")


main()

students.txt not found. Starting with empty data.
courses.txt not found. Starting with empty data.
registrations.txt not found. Starting with empty data.
Loaded 2 students and 4 courses.

===== COURSE REGISTRATION PORTAL =====
1. Admin login
2. Student login
3. Exit


Choose:  2
Username / Student ID:  s001
Password:  pass123


Wrong ID or password.

===== COURSE REGISTRATION PORTAL =====
1. Admin login
2. Student login
3. Exit


Choose:  2
Username / Student ID:  S001
Password:  pass123



Welcome, Ayesha Khan

--- STUDENT MENU ---
1. View courses
2. Search courses
3. Register in a course
4. Drop a course
5. My courses
6. Logout


Choose:  1



CODE    TITLE                         CREDITS  SEATS LEFT
CS101 | Programming Fundamentals | 3 credits | 3 seats left
MG201 | Principles of Management | 3 credits | 30 seats left
ST301 | Statistics for Analytics | 3 credits | 30 seats left
MK210 | Marketing Basics | 2 credits | 25 seats left

--- STUDENT MENU ---
1. View courses
2. Search courses
3. Register in a course
4. Drop a course
5. My courses
6. Logout


Choose:  6


Logged out.

===== COURSE REGISTRATION PORTAL =====
1. Admin login
2. Student login
3. Exit


Choose:  3


Goodbye!
